# Increasing signal to noise ratio via integration gain 

Consider a signal of the form: 

$$
s = A B(t) + n(t)
$$

Where:

n is a gaussian distributed noise with unity variance. 

$ n \sim N( 0 , 1 ) $

$B(t)$ is a sequence of bits at a specific bit rate $F_b$

A is the amplitude of the signal calculated based on the unity noise variance and a desired carrier to noise density ratio $CN0$


$ A = $

Below we will generate this signal and give you the opportunity to change CN0 with a slider. Note that the bits change every bit period so that you can clearly see the edges. 


In [90]:
import numpy as np 
import plotly.graph_objects as go
import ipywidgets as widgets

In [91]:
# Sample rate 
Fs = 1.0e4
# Bit Rate 
Fb = 10
# total time 
T = 1.0
seed = 42



In [92]:
def makeSignal( CN0 ):
    global t, signal
    
    rng = np.random.default_rng(seed=seed)                  # seed is optional, for reproducibility
    L = int( Fs * T )

    # Translate CN0 to variance and std

    var = 1.0 # unity variance
    std = np.sqrt( var )
    noise = rng.normal( loc=0.0, scale=std, size=L)
    # bit signal
    samplesPerBit = int( Fs / Fb )
    nBits = L / samplesPerBit
    bits = np.arange( nBits ) % 2
    symbols = 2*bits-1
    A = np.sqrt( 2 * 10**( CN0 / 10 ) / Fs ) # unity variance -> N0 = 2 / Fs, C = A^2 = CN0 * N0 
    signal = A * np.repeat(symbols, samplesPerBit)

    # Time
    t = np.arange( L ) / Fs



    signal = noise + signal

    # Plot the signal
    # t vs signal
    fig = go.Figure()
    fig.add_trace( go.Scattergl( x=t, y=signal, mode="lines", name="signal" ) )
    fig.update_layout( title=f"Signal - CN0 = {CN0:g} dB-Hz", xaxis_title="t (s)", yaxis_title="amplitude" )
    return fig

# N is a slider for number of samples to integrate over 
def plotCoherent( t , signal, CN0, N ):
    B = len(signal) // N                # number of whole blocks of N samples
    signalClipped = signal[:B*N]
    signalCoherent = np.reshape( signalClipped, (B, N) )
    coherent = np.sum( signalCoherent, axis=1 ) /  N
    tClipped = t[:B*N]
    tCoherent = tClipped[::N] # every Nth sample 

    # plot signal and coherent integration signal so we can see
    # - down sampling 
    # - gain
    fig = go.Figure()
    fig.add_trace( go.Scattergl( x=t, y=signal, mode="lines", name="signal" ) )
    fig.add_trace( go.Scattergl( x=tCoherent, y=coherent, mode="markers", name="coherent" ) )
    fig.update_layout( title=f"Coherent Integration - CN0 = {CN0:g} dB-Hz, N = {N}", xaxis_title="t (s)", yaxis_title="amplitude" )
    return fig

# M is a slider for number of coherent intervals to sum over 
def plotNonCoherent( t, signal, CN0, N, M ):
    # coherent integration over N samples
    B = len(signal) // N                # number of whole blocks of N samples
    signalClipped = signal[:B*N]
    coherent = np.sum( np.reshape( signalClipped, (B, N) ), axis=1 ) / N
    tCoherent = t[:B*N][::N] # every Nth sample 

    # non-coherent integration: square the coherent outputs, then sum M of them
    K = B // M                          # number of whole groups of M coherent outputs
    power = coherent[:K*M]**2
    nonCoherent = np.sum( np.reshape( power, (K, M) ), axis=1 ) / M
    tNonCoherent = tCoherent[:K*M][::M] # every Mth coherent sample

    # plot signal, coherent and non-coherent signal so we can see
    # - squaring removes the bit sign
    # - summing M squared outputs smooths out the noise
    fig = go.Figure()
    fig.add_trace( go.Scattergl( x=t, y=signal, mode="lines", name="signal" ) )
    fig.add_trace( go.Scattergl( x=tCoherent, y=coherent, mode="markers", name="coherent" ) )
    fig.add_trace( go.Scattergl( x=tNonCoherent, y=nonCoherent, mode="markers", name="non-coherent" ) )
    fig.update_layout( title=f"Non-Coherent Integration - CN0 = {CN0:g} dB-Hz, N = {N}, M = {M}", xaxis_title="t (s)", yaxis_title="amplitude" )
    return fig

In [93]:
# carrier to noise density - dB/Hz 
CN0 = widgets.FloatSlider( value=35, min=0, max=100, step=1, description="CN0 (dB-Hz)", continuous_update=False )

# one plot, redrawn in place when the slider moves
display( CN0 )
# pass plotly's mimebundle directly - display(fig) calls fig.show() and ignores display_id
plot = display( makeSignal( CN0.value )._repr_mimebundle_(), raw=True, display_id=True )
CN0.observe( lambda change: plot.update( makeSignal( change.new )._repr_mimebundle_(), raw=True ), names="value" )

FloatSlider(value=35.0, continuous_update=False, description='CN0 (dB-Hz)', step=1.0)

Notice how as you raise $CN0$ the bits become more and more clear. But for some values like $CN0=35 dB-Hz$ the are many samples where the bit value is on the wrong side of zero. 

## Coherent Integration

To properly find the bits we want to raise the signal to noise ratio of the signal and we can do this via a very simple digital signal processing method - integration.

To understand why this works we will have to cover some basic math. Recall that:

$$
n \sim N(0,1)
$$

The discrete signal of interest is:

$$
s_k = B_k A  + n 
$$

$B_k$ can only ever be $\pm 1$ - so we'll simplify that to 1
$A$ is constant based on CN0 

With this simplification 

$$
s \sim N(A,1)
$$

The signal to noise ratio of a signal is signal power over noise power. 

$$
SNR = \frac{A^2}{\sigma^2}
$$

Since $\sigma_s=1$ that means:

$$
SNR_s = \frac{A^2}{1} = A^2 
$$

Now we can introduce integration. Stastics tells us the following rule

If 
$$
a_k \sim N(A,\sigma^2)
$$

Then 
$$
b = \sum^n \left( a_k \right) \sim N( nA , n\sigma^2 )
$$

And following this we can calculate:

$$
SNR_b = \frac{ (nA)^2 }{ (n\sigma^2)} = \frac{ n^2 A^2 }{ n \sigma^2 } = \frac{nA}{\sigma^2} = n SNR_a
$$

In short summation / integration raises the signal to noise ratio of a signal by a factor of $n$ - the number of samples being summed over

In [100]:
# coherent integration length - samples
# only divisors of samplesPerBit, so no block of N samples straddles a bit edge
samplesPerBit = int( Fs / Fb )
validN = [ n for n in range( 1, samplesPerBit + 1 ) if samplesPerBit % n == 0 ]
N = widgets.SelectionSlider( options=validN, value=validN[4], description="N (samples)", continuous_update=False )

# one plot, redrawn in place when the slider moves
display( N )
coherentPlot = display( plotCoherent( t, signal, CN0.value, N.value )._repr_mimebundle_(), raw=True, display_id=True )
N.observe( lambda change: coherentPlot.update( plotCoherent( t, signal, CN0.value, change.new )._repr_mimebundle_(), raw=True ), names="value" )

SelectionSlider(continuous_update=False, description='N (samples)', index=4, options=(1, 2, 4, 5, 8, 10, 20, 2…

## Non Coherent Integration 

So if we can just sum to raise SNR why dont we just integrate forever. 

Reasons 
First signals change frequence due to motion (doppler) and signal tracking loops need to return a value
More importantly - the pesky data
- The bits....they are actually +/-1 

Consider a signal of length 2000 where we no longer assume  $B(t)=1$ but rather 
$$
B =
\begin{cases}
1 & \text{for } k < 1000 \\
-1 & \text{for } k > 1000
\end{cases}
$$

In this case lets see what happens if we try to integrate for longer than 1000 samples

$$
s_k = A B_k + n_k 
$$

The summed signal $x_k$

$$
x_k = \sum_0^n=2000\left( s_k \right) = \sum_{k=0}^{999}(s_k) + \sum_{k=1000}^{1999}(s_k)
$$


TODO equation that show that this basically equates to adding 1 and -1 together and they cancel out


So what do we do - well if you need to preserve the bits for decoding there isnt much more to do. However, if you want to drive a tracking loop you may not need the bits to do this in which case you actually can integrate across the bit boundaries.

But how. Well if we coherently integrate the signal...then square it...then add the squared signals we can continue to see SNR growth without interference from teh bit edges. 

So now lets recall that 

$n$ is the number of samples coherently integrated - we will try to maximize this at the bit length
$m$ is the number of non coherent integrations. So lets do the math and see what happens

If:

$$
s_k \sim N(A , \sigma^2 )
$$

then the coherent sum is also a normal distribution 

$$
b_k = \sum_{k=0}{n}( s_k ) \sim N( nA, n\sigma^2 )
$$

but now we introduce the square of this distribution. The square of a normal distribution is $\chi^2$

$$
c_k = \left| b_k \right|^2 \sim \chi^2(  n^2 A^2 + n \sigma^2 , 4 n^3 A^2 \sigma^2 + 2 n^2 \sigma^4 )
$$

And this means that

$$
SNR_{c} = \frac{ (n^2 A^2 + n \sigma^2) ^ 2 }{4 n^3 A^2 \sigma^2 + 2 n^2 \sigma^4} = \frac{ n^2 ( n A^2 + \sigma^2)^2 }{ 2n^2\sigma^2(2nA^2+\sigma^2)}
= \frac{ ( n A^2 + \sigma^2)^2 }{ 2\sigma^2(2nA^2+\sigma^2)}
$$

And finally by summing 

$$
d_k = \sum^m( c_k ) \sim \xi^2 \left( mn^2A^2 + mn\sigma^2 , 4 m n^3 A^2 \sigma^2 + 2 m n^2 \sigma^4 \right)
$$

Then 

$$
SNR_d = m SNR_c  = m \frac{ ( n A^2 + \sigma^2)^2 }{ 2\sigma^2(2nA^2+\sigma^2)}
$$


This is an improvement in resolution compared to $b_k$ 

To demonstrate this use the slider below to raise non-coherent integration factor. You'll notice that **the bits are destroyed** but that the amount of noise on the remaining signal goes down as $m$ goes up.

In [ ]:
# non-coherent integration length - number of coherent intervals
M = widgets.IntSlider( value=1, min=1, max=10, step=1, description="M (intervals)", continuous_update=False )

# one plot, redrawn in place when either slider moves
display( M )
nonCoherentPlot = display( plotNonCoherent( t, signal, CN0.value, N.value, M.value )._repr_mimebundle_(), raw=True, display_id=True )
redrawNonCoherent = lambda change: nonCoherentPlot.update( plotNonCoherent( t, signal, CN0.value, N.value, M.value )._repr_mimebundle_(), raw=True )
M.observe( redrawNonCoherent, names="value" )
N.observe( redrawNonCoherent, names="value" )

IntSlider(value=1, continuous_update=False, description='M (intervals)', max=10, min=1)

## Probability of detection and false alarm